# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://python.langchain.com/docs/integrations/document_loaders/
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

## 주요 Document Loader

### Text file
- TextLoader 이용

In [1]:
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

with open(path, "rt", encoding="utf-8") as f:
    print(f.read()[:100])

C:\Users\Playdata\AppData\Local\Temp\ipykernel_17876\1201840191.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하


In [5]:
# 다른 방식 (TextLoader를 이용해서 조회)
## 객체 생성 - 읽을 파일의 경로를 지정.
loader = TextLoader(path, encoding='utf-8')

# 읽어오기
docs = loader.load() # 실행 시 바로 읽는다. => 반환: List[Document]
# docs = loader.lazy_load() # 읽은 문서를 사용할 때 읽는다. 반환: generator[Document]

print(type(docs), len(docs))
print(type(docs[0]))

# for doc in docs: # lazy_load() -> generator
#     print(type(doc))


<class 'list'> 1
<class 'langchain_core.documents.base.Document'>


In [ ]:
doc = docs[0]
print("문서정보: doc.metadata")
print(doc.metadata)
# 필요한 정보들을 추가할 수 있다. LLM에 전달할 프롬프트에 추가할 정보. 검색할 때 사용할 정보들.
# 메타데이터의 키 -> 사전에 설계가 필요.
doc.metadata['category'] = "sports"
doc.metadata['tag'] = ["올림픽", "IOC", "동계올림픽", "하계올림픽"]

print(doc.metadata)

문서정보: doc.metadata
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}


In [11]:
print("문서내용: doc.page_content")
print(doc.page_content[:200])

문서내용: doc.page_content
올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열


In [12]:
print("문서 식별자-id: doc.id")
print(doc.id)

문서 식별자-id: doc.id
None


### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [30]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

path= "data/novel/금_따는_콩밭_김유정.pdf"

# Document Loader 생성
# loader = PyPDFLoader(path, mode= "single") # mode: single(전체를 하나의 문서로 읽기)
# loader = PyPDFLoader(path, mode= "page") # mode: page(페이지당 하나의 문서로 읽기)
loader = PDFPlumberLoader(path)

## PyMuPDF
# loader = PyMuPDFLoader(path)

# 읽기
docs = loader.load() # list[Document]
print(len(docs))

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

23


In [27]:
from pprint import pprint
doc = docs[0]
pprint(doc.metadata)
# doc.metadata['author'] = '김유정'

{'author': 'Unknown',
 'creationdate': '2024-11-24T07:05:35+00:00',
 'creator': 'Wikisource',
 'moddate': '2024-11-24T07:05:37+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Wikisource',
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'title': '금 따는 콩밭',
 'total_pages': 23}


In [17]:
print(doc.page_content)

1
금  따는  콩밭
Exported from Wikisource on 2024 년  11 월  24 일
2
위키백과
위키백과에  이  글
과  관련된 
자료가  있습니다 .
금  따는  콩밭
🙝🙟
땅속  저  밑은  늘  음침하
다 .
고달픈  간드렛불 , 맥없이
푸르끼하다 .
밤과  달라서  낮엔  되우  흐릿하였다 .
겉으로  황토  장벽으로  앞뒤좌우가  콕  막힌  좁직한  구뎅이 .
흡사히  무덤  속같이  귀중중하다 . 싸늘한  침묵 , 쿠더브레한
흙내와  징그러운  냉기만이  그  속에  자욱하다 .
곡괭이는  뻔질  흙을  이르집는다 . 암팡스러이  내려쪼며 ,
퍽  퍽  퍼억 .
이렇게  메떨어진  소리뿐 . 그러나  간간  우수수  하고  벽이  헐
린다 .
영식이는  일손을  놓고  소맷자락을  끌어당기어  얼굴의  땀을
훑는다 . 이놈의  줄이  언제나  잡힐는지  기가  찼다 . 흙  한줌을
집어  코밑에  바짝  들여대고  손가락으로  샅샅이  뒤져본다 . 완
연히  버력은  좀  변한  듯싶다 . 그러나  불통버력이  아주  다  풀
린  것도  아니었다 . 밀똥버력이라야  금이  온다는데  왜  이리
안  나오는지 .
곡괭이를  다시  집어든다 . 땅에  무릎을  꿇고  궁뎅이를  번쩍
든  채  식식거린다 . 곡괭이는  무작정  내려찍는다 . 바닥에서
3
물이  스미어  무르팍이  흔건히  젖었다 . 굿엎은  천판에서  흙방
울은  내리며  목덜미로  굴러든다 . 어떤  때에는  웃벽의  한쪽이
떨어지며  등을  탕  때리고  부서진다 .
그러나  그는  눈도  하나  깜짝하지  않는다 . 금을  캔다고  콩밭
하나를  다  잡쳤다 . 약이  올라서  죽을둥  살둥  눈이  뒤집힌  이
판이다 . 손바닥에  침을  탁  뱉고  곡괭이  자루를  한번  꼰아잡
더니  쉴  줄  모른다 .
등뒤에서는  흙  긁는  소리가  드윽드윽  난다 . 아직도  버력을
다  못  친  모양 . 이  자식이  일을  하나  시졸  하나 . 

### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [42]:
from bs4 import BeautifulSoup

html_txt = """<html>
<body>
<p><b>제목</b> <span>내용</span></p>
<p>다음 문단</p>
<div>다음 내용</div>
</body>
</html>
"""

soup = BeautifulSoup(html_txt)
txt1 = soup.get_text() # 태그내의 text(content)만 추출한다.
txt2 = soup.get_text(strip=True)
txt3 = soup.get_text().strip()
txt4 = soup.get_text(strip=True, separator="\n")
print("==============기본===============")
print(txt1)
print("==============Strip=True===============")
print(txt2)
print("===============.Strip()================")
print(txt3)
print("===============Strip=True, Separator=\\n==============")
print(txt4)

==============기본===============


제목 내용
다음 문단
다음 내용



==============Strip=True===============
제목내용다음 문단다음 내용
===============.Strip()================
제목 내용
다음 문단
다음 내용
===============Strip=True, Separator=\n==============
제목
내용
다음 문단
다음 내용


In [ ]:
from bs4 import SoupStrainer

strainer = SoupStrainer("span") # <span>
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<span>내용</span>

In [44]:
from bs4 import SoupStrainer

strainer = SoupStrainer("p") # <span>
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<p><b>제목</b> <span>내용</span></p><p>다음 문단</p>

In [45]:
from bs4 import SoupStrainer

strainer = SoupStrainer(["p", "div"]) # <span>
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<p><b>제목</b> <span>내용</span></p><p>다음 문단</p><div>다음 내용</div>

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import os

os.environ['USER_AGENT'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"

url = [
    "https://m.sports.naver.com/wbaseball/article/311/0002025983",
    "https://m.sports.naver.com/general/article/108/0003447587"
]

loader = WebBaseLoader(
    # web_path= url[1], # 1개 문서 조회
    web_path = url, # 여러개 문서 조회 list["url"들]
    # default_parser="lxml", # BeautifulSoup(문서str, parser(html.parser))
    bs_kwargs={ # BeautifulSoup() 생성할 때 넣어줄 파라미터 설정.
        "parse_only":SoupStrainer(name="div", attrs={"class":"_article_content"})
    },
    bs_get_text_kwargs={
        "strip":True
        "separator":"\n\n"
    }
    )

In [52]:
docs = loader.load()
print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content)

2
{'source': 'https://m.sports.naver.com/wbaseball/article/311/0002025983', 'title': '[속보] 이정후, 홈런 포함 멀티히트 쾅쾅!…방망이 완전 부활→타율 1위 맹추격+최근 4경기 3번째 멀티히트', 'language': 'ko'}
[속보] 이정후, 홈런 포함 멀티히트 쾅쾅!…방망이 완전 부활→타율 1위 맹추격+최근 4경기 3번째 멀티히트본문 바로가기NAVER스포츠뉴스엔터메뉴북중미 월드컵홈야구해외야구축구해외축구농구배구N골프일반e스포츠아웃도어NEW뉴스영상일정순위포토홈 바로가기NAVER스포츠뉴스엔터스포츠북중미 월드컵야구해외야구축구해외축구농구배구N골프일반e스포츠아웃도어콘텐츠오늘의 경기승부예측연재이슈톡대학스포츠랭킹기타고객센터공식 블로그메뉴 닫기[속보] 이정후, 홈런 포함 멀티히트 쾅쾅!…방망이 완전 부활→타율 1위 맹추격+최근 4경기 3번째 멀티히트입력2026.06.24. 오후 12:04기사원문공감좋아요0슬퍼요0화나요0팬이에요0후속기사 원해요0텍스트 음성 변환 서비스본문 듣기를 종료하였습니다.글자 크기 변경공유하기미국 메이저리그(MLB) 샌프란시스코 자이언츠의 이정후가 첫 타석부터 홈런에 이은 멀티히트를 폭발했다. 샌프란시스코 자이언츠는 24일 오전 10시 45분부터 미국 캘리포니아주 샌프란시스코의 오라클 파크에서 애슬레틱스와 2026 메이저리그 정규시즌 홈 경기를 치르는 중이다. 이정후는 2회말 1사 주자 없는 상황에서 첫 타석에 등장한 뒤 상대 선발 애런 서발리의 2구를 통타해 솔로포를 터트렸다. 자신의 시즌 5호 홈런이다. 이어 4회말엔 내야안타로 멀티히트를 일찌감치 기록했다. 연합뉴스  (엑스포츠뉴스 이우진 기자) 미국 메이저리그(MLB) 샌프란시스코 자이언츠의 이정후(27)가 대형 홈런에 멀티히트까지 터뜨리며 타율왕 경쟁에 다시 불을 지폈다.샌프란시스코는 24일 오전 10시 45분(한국시간)부터 미국 캘리포니아주 샌프란시스코의 오라클 파크에서 애슬레틱스와 2026 메이저리그 정규시즌 홈 경기를

#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
  ```bash
  시작 URL
  ├─ 내부 링크 1
  │   ├─ 내부 링크 1-1
  │   └─ 내부 링크 1-2
  ├─ 내부 링크 2
  └─ 내부 링크 3
  ```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
    ```

In [61]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

start_url = "https://docs.python.org/3"

def extractor(html:str) -> str:
    """RecursiveUrlLoader는 HTML문서를 그대로 반환.
    HTML 문서를 str으로 받아서 원하는 부분만 추출하는 callback함수
    Args:
        html(str): HTML(웹) 문서
    Returns:
        str: html에서 body의 content(text)만 추출해서 반환.
    """
    soup = BeautifulSoup(html)
    body_element = soup.select_one("div.body") # python doc 사이트 - 내용이 <div class='body'> 내에 위치
    return body_element.get_text(strip=True, separator="\n") if body_element is not None else soup.get_text(strip=True, separator="\n")

loader = RecursiveUrlLoader(
    url = start_url,
    max_depth= 2, #0: start_url, 1: start_url -> link, 2: start_url -> link -> link
    prevent_outside=True,
    base_url = start_url,
    extractor= extractor
)

docs = loader.load()

In [62]:
len(docs)

19

In [ ]:
idx = 1
doc = docs[idx]
doc.metadata
# pprint(doc.metadata)

{'source': 'https://docs.python.org/3/index.html',
 'content_type': 'text/html',
 'title': '3.14.6 Documentation',
 'description': 'The official Python documentation.',
 'language': 'en'}

In [ ]:
# for doc in docs:
#     print(doc.metadata['source'])

In [65]:
print(doc.page_content)

Python 3.14.6 documentation
Welcome! This is the official documentation for Python 3.14.6.
Documentation sections:
What's new in Python 3.14?
Or
all "What's new" documents since Python 2.0
Tutorial
Start here: a tour of Python's syntax and features
Library reference
Standard library and builtins
Language reference
Syntax and language elements
Python setup and usage
How to install, configure, and use Python
Python HOWTOs
In-depth topic manuals
Installing Python modules
Third-party modules and PyPI.org
Extending and embedding
For C/C++ programmers
Python's C API
C API reference
FAQs
Frequently asked questions (with answers!)
Deprecations
Deprecated functionality
Other resources:
Python Packaging User Guide
Resources relating to Python packaging
Static Typing with Python
Information and guides about Python type safety
Indices, glossary, and search:
Global module index
All modules and libraries
General index
All functions, classes, and terms
Glossary
Terms explained
Search page
Search this

### <del>ArxivLoader</del>

- arxiv api가 업데이트 되고 ArxivLoader는 그에 맞춰 업데이트가 되지 않아 ArxivLoader는 정상적으로 실행되지 않는다. **arxiv api 를 이용해서 검색 후 pdf를 다운로드 받는다.**
- arxiv API: https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- 설치
  - `pip install arxiv`



In [66]:
# arxiv lib를 이용해 arxiv.org의 눈문을 검색해 다운로드 or 논문의 주요 정보 조회
# RAG 문서로 arxiv의 논문들이 필요할 경우 이용 가능
import arxiv
# 검색관련 설정
search = arxiv.Search(
    query="Advanced RAG",  # 검색어
    max_results=10,  # 검색 논문 최대 개수
    sort_by=arxiv.SortCriterion.Relevance  # 정렬 기준
)
# 정렬 기준
arxiv.SortCriterion
## .Relevance: query와 관련성 높은(정확도) 순서
## .LastUpdatedDaet: 논문이 마지막으로 수정 업데이트된 날짜
## .SubmittedDate: 논문이 처음 제출된 날짜
# 검색
client = arxiv.Client()
results = client.results(search)
print(type(results)) # iterator 타입

<class 'itertools.islice'>


In [67]:
paper = next(results) # 첫번째 논문
print(type(paper))

print(paper.title) # 논문 제목
print(paper.authors) # 논문 저자들 list[Author]
print(paper.authors[0].name) # 첫번째 저자의 이름
print(paper.categories) # 논문의 분야(카테고리)
print(paper.summary) # 논문 요약 내용
print(paper.pdf_url) # 논문 PDF 파일의 url
print(paper.published) # 논문 발표 일시
print(paper.get_short_id()) # 논문 ID

<class 'arxiv.Result'>
MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries
[arxiv.Result.Author('Yixuan Tang'), arxiv.Result.Author('Yi Yang')]
Yixuan Tang
['cs.CL']
Retrieval-augmented generation (RAG) augments large language models (LLM) by retrieving relevant knowledge, showing promising potential in mitigating LLM hallucinations and enhancing response quality, thereby facilitating the great adoption of LLMs in practice. However, we find that existing RAG systems are inadequate in answering multi-hop queries, which require retrieving and reasoning over multiple pieces of supporting evidence. Furthermore, to our knowledge, no existing RAG benchmarking dataset focuses on multi-hop queries. In this paper, we develop a novel dataset, MultiHop-RAG, which consists of a knowledge base, a large collection of multi-hop queries, their ground-truth answers, and the associated supporting evidence. We detail the procedure of building the dataset, utilizing an English 

In [68]:
# 논문 pdf파일 다운로드
import os
import requests

save_dir = "data/papers"
os.makedirs(save_dir, exist_ok=True)

resp = requests.get(paper.pdf_url)
if resp.status_code == 200:
    with open(os.path.join(save_dir, paper.get_short_id()+".pdf"), "wb") as fo:
        fo.write(resp.content)

In [69]:
# 논문 pdf 파일 다운로드 함수
import os
import requests

def download_arxiv_paper(paper:arxiv.Result, dirpath:str):
    """검색결과(paper)를 dirpath에 저장."""
    os.makedirs(dirpath, exist_ok=True)

    resp = requests.get(paper.pdf_url)
    if resp.status_code == 200:
        with open(os.path.join(dirpath, paper.get_short_id()+".pdf"), "wb") as fo:
            fo.write(resp.content)

In [ ]:
save_dir = "data/papers"

for paper in results:
    download_arxiv_paper(paper, save_dir)

In [83]:
# arxiv에서 검색어의 논문을 조회해서 다운 받은 후에
# Document에 page_content에는 논문 내용을 metadata에는 위의 정보를 넣어서
# list[Document]를 반환하는 함수.

import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

def load_arxiv_docs(
        query: str, # 검색어
        top_k: int= 10, # 최대 검색 개수
        dirpath: str="." # 논문 저장할 디렉토리
)-> list[Document]:
    client = arxiv.Client()
    search = arxiv.Search(
        query= query, max_results= top_k, sort_by= arxiv.SortCriterion.Relevance
    )
    
    results = client.results(search)
    docs = [] # 생성한 Document(조회결과)를 담은 리스트
    for paper in results:
        # 다운로드
        download_arxiv_paper(paper, dirpath)
        # PDF 문서 load
        file_path = os.path.join(dirpath, paper.get_short_id()+".pdf")
        loader = PyPDFLoader(file_path, mode="single")
        doc:Document = loader.load()[0] # list[Document]
        # 메타데이터는 paper의 정보로 변경.
        doc.metadata = {
            "title":paper.title,
            "authors":[a.name for a in paper.authors],
            "categories": paper.categories,
            "arxiv_url": paper.entry_id, # 논문 페이지 url
            "pdf_url": paper.pdf_url
        }
        docs.append(doc)

    return docs

In [84]:
docs = load_arxiv_docs(query="transformers", top_k= 20, dirpath="data/papers/transformers")

PdfReadError("Invalid Elementary Object starting with b'\\\\' @756840: b'ateVersion (2021.1) \\\\par /Author()/Title()/Subject()/Creator(LaTeX with hyperref'")
Multiple definitions in dictionary at byte 0xb8c76 for key /Author
Multiple definitions in dictionary at byte 0xb8c7e for key /Title


In [85]:
print(len(docs))

20


In [86]:
print(type(docs))

<class 'list'>


In [87]:
docs[0].metadata

{'title': 'Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants',
 'authors': ['Jose I. Aizpurua'],
 'categories': ['cs.LG'],
 'arxiv_url': 'http://arxiv.org/abs/2512.22190v1',
 'pdf_url': 'https://arxiv.org/pdf/2512.22190v1'}

### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [ ]:
# GPU가 있는 경우 -> torch 부터 gpu 버전으로 설치.

In [4]:
# Huggingface 로그인 - docling이 사용하는 모델은 로그인 후에 받을 수 있다.
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

access_key = os.getenv("HUGGINGFACE_API_KEY")
login(access_key)

In [12]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# 현재 torch와 docling이 사용하는 OCR Lib (RapidOCR) 호환성 문제 때문에 OCR 기능을 끄는 설정을 한다.
pdf_option = PdfPipelineOptions()
pdf_option.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options= pdf_option
        )
    }
)

path = ["data/papers/2409.03708v2.pdf", "data/papers/transformers/2010.12698v2.pdf"]
loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN,
    converter= converter
)

In [13]:
docs = loader.load()

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [15]:
doc = docs[1]
doc.metadata

{'source': 'data/papers/transformers/2010.12698v2.pdf'}

In [16]:
print(doc.page_content)

## Stabilizing Transformer-Based Action Sequence Generation For Q-Learning

## Gideon Stein 1 , Andrey Filchenkov 2 , Arip Asadulaev 3

ITMO University St. Petersburg, 198215

1 gideon@steinml.de, 2 afilchenkov@itmo.ru, 3 aripasadulaev@itmo.ru

## Abstract

Since the publication of the original Transformer architecture (Vaswani et al. 2017), Transformers revolutionized the field of Natural Language Processing. This, mainly due to their ability to understand timely dependencies better than competing RNN-based architectures. Surprisingly, this architecture change does not affect the field of Reinforcement Learning (RL), even though RNNs are quite popular in RL, and time dependencies are very common in RL. Recently, (Parisotto et al. 2019) conducted the first promising research of Transformers in RL. To support the findings of this work, this paper seeks to provide an additional example of a Transformer-based RL method. Specifically, the goal is a simple Transformer-based Deep Q-Learning 

In [17]:
from IPython.display import Markdown
Markdown(doc.page_content)

## Stabilizing Transformer-Based Action Sequence Generation For Q-Learning

## Gideon Stein 1 , Andrey Filchenkov 2 , Arip Asadulaev 3

ITMO University St. Petersburg, 198215

1 gideon@steinml.de, 2 afilchenkov@itmo.ru, 3 aripasadulaev@itmo.ru

## Abstract

Since the publication of the original Transformer architecture (Vaswani et al. 2017), Transformers revolutionized the field of Natural Language Processing. This, mainly due to their ability to understand timely dependencies better than competing RNN-based architectures. Surprisingly, this architecture change does not affect the field of Reinforcement Learning (RL), even though RNNs are quite popular in RL, and time dependencies are very common in RL. Recently, (Parisotto et al. 2019) conducted the first promising research of Transformers in RL. To support the findings of this work, this paper seeks to provide an additional example of a Transformer-based RL method. Specifically, the goal is a simple Transformer-based Deep Q-Learning method that is stable over several environments. Due to the unstable nature of Transformers and RL, an extensive method search was conducted to arrive at a final method that leverages developments around Transformers as well as Q-learning. The proposed method can match the performance of classic Qlearning on control environments while showing potential on some selected Atari benchmarks. Furthermore, it was critically evaluated to give additional insights into the relation between Transformers and RL.

Transformer architectures revolutionized the field of Natural Language Processing (NLP). The classic Transformer (Vaswani et al. 2017) and its successors such as (Devlin et al. 2018) or (Radford et al. 2019) outperform traditionally used RNN-based architectures on the majority of tasks in NLP. Their superior performance can be mostly attributed to their ability to understand timely dependencies and notably longterm dependencies better than RNN-based methods. Timely dependencies are not only interesting for NLP tasks. In reinforcement learning, such an ability is useful to perform in environments that are only partially observable. RNN-based methods are traditionally deployed to such environments. Furthermore, there are multiple successful examples of applications of the Attention mechanism (the core functionality of the Transformer) to RL (Iqbal and Sha 2019), (Oh et al. 2016) and (Manchin, Abbasnejad, and van den Hengel 2019). Due to these facts, the deployment of Transformerbased architectures to RL is a promising research direction. While there were several studies on Transformer-based methods in RL, many of them such as (Upadhyay et al. 2019) or (Mishra et al. 2017) reported random performance for their Transformer-based approaches. Even on simple MDP or Multi-armed Bandit problems. Contrary to that, the first major success with Transformer-based RL methods was recently reported by (Parisotto et al. 2019). So while it is possible to use Transformer-based architectures in RL, it seems to be a nontrivial task. In RL, the information signal is affected by past decisions. This creates dependencies and makes optimization harder than training on a fixed dataset. Additionally, Transformer-architectures are quite hard to optimize which was already stated in (Vaswani et al. 2017). As an example, they are strongly dependent on a specific learning rate schedule to be optimized. Together, these two conditions make the optimization of Transformer-based architectures in RL challenging.

To support the results of (Parisotto et al. 2019) and to further evaluate the possibility of Transformer-based models in RL, this paper seeks to create a new Transformerbased RL method that contrary to (Parisotto et al. 2019) features a model that is based on the original Transformer from (Vaswani et al. 2017) instead of the Transformer-XL (Dai et al. 2019). Furthermore, this work will alternatively use the Transformer-based model as a value function for a Deep Qlearning agent. While the approach of (Parisotto et al. 2019) showed strong performance, its optimization method as well as their Transformer model are quite advanced. On contrary, the approach in this paper uses a well-known optimization method (Q-learning) as well as a simple Transformer version to add to a clear understanding of the interrelations between RL and Transformer-based models. Furthermore, it is hoped that this choice will make it easier to retrace results that are reported in this paper and encourage additional research. In summary, the following ideas are hoped to be supported:

- With a couple of alterations, Transformers can generally perform in RL.
- Transformer-based Deep Q-networks (TBQN) can perform.
- Transformers can outperform RNN based models in RL.

## Background

## Deep Q-learning

The goal of Q-learning is to find a function that correctly maps state (s) action (a) pairs to their corresponding value for an agent that interacts with an environment. The simplest form of Q-learning is defined as updating a Q-function in the following manner:

<!-- formula-not-decoded -->

where R is a reward, γ is a discount factor and Q represents the value of any pair (s,a). By updating Q-values after rewards, the greedy policy as-well as the value function change frequently. Under the condition that all states are explored sufficiently, these updates are guaranteed to converge to a Q-function that correctly represents the environment. Based on this Q-function, a policy will be formed by greedily sampling the action with the highest Q-value at every step. Traditionally, the Q-function was implemented as a table. However, it is possible to approximate it with a neural network. This method is known as Deep Q-learning. When using a function approximator for the Q-function, the definition of an update changes since it is only possible to update weights of the network and not specific Q-values directly. An update for a Deep Q-network (DQN) is therefore defined as:

<!-- formula-not-decoded -->

where θ represents the network weights and s ′ and a ′ are the state and action in the timestep t+1. When a network is updated according to Formula 2, an issue arises. Data that is acquired by an agent interacting with an environment is quite different from a fixed dataset that is normally used to train neural networks. Today, Replay Buffers, a method to save experience and reuse it during model training, and target networks, a method to make the target of the update more stable, are used to counter these issues. By applying these two methods to Deep Q-learning, (Mnih et al. 2013) opened the field of Deep Reinforcement Learning. Their method will be the baseline RL method for the course of this work.

## The Transformer Architecture

The Transformer model is a sequence to sequence architecture (seq2seq) which was initially developed to perform translation tasks in NLP, and which relies heavily on the Attention mechanism. A seq2seq structure is defined as a model that takes in a sequence of signals and returns a sequence of outputs. Also, the Transformer is an EncoderDecoder structure that splits into two distinct submodels. An Encoder, that transforms an input sequence into an encoded representation and a Decoder that generates, based on the encoded representation, a new sequence as an output. Since the proposed architecture is based on the Encoder of the classic Transformer (Vaswani et al. 2017), its structure will be discussed further. The Transformer Encoder takes in some word tokens and transforms them into the same number of encoded representations. To do this efficiently, the Transformer stacks several identical blocks on top of each other. These blocks are called Encoder layers. Additionally, the Encoder features an Embedding layer and a positional encoding of the input sequences which both are added before the first layer. A single Encoder layer is constructed out of two main components. An Attention block and a feed-forward network. Additionally, residual connections and normalization are added. Fig. 1 shows the structure of the Encoder layer. Note that the input and the output of the Encoder have the same dimension. This makes layer stacking possible. The computation that takes place in a single Encoder layer is defined as:

Figure 1: The standard Transformer Encoder layer

<!-- formula-not-decoded -->

where X represents an input tensor with the shape (batch size, input sequence length, model dimension). Typically, Dropout is deployed after the Attention block and after the feed-forward block.

## The Attention mechanism

Attention is a mechanism that understands the importance of specific inputs for other inputs and combines these into a new vector that includes this information. This mechanism does not rely on a hidden representation that includes all past information but attends directly to the full inputs. This helps Attention to perform better than RNN-based approaches in many cases, especially when long-term dependencies are present and relevant. Based on Attention, Multi-Head Attention is performed by multiple Attention operations in parallel on sub-parts of the inputs. This allows attending multiple sub-areas of inputs at once. The Transformer features the use of Scaled Dot Product Attention as well as Multi-Head Attention to understand dependencies. Scaled dot product Attention is defined as the operation on three inputs. Keys (K), Queries (Q), and Values (V):

<!-- formula-not-decoded -->

Where Q, K, V are input matrices, and dim key is the last dimension of K. Intuitively, this can be understood as a way to scale and add the content of V by a factor that is a combination of Q and V. Through this channel, V attends to the information that is included in Q and K and is altered accordingly. To perform Multi-Head Attention, the initial input vector is simply split. When performing Multi-Head Attention in the Encoder, the embedded input sequence represents K, Q, and V. This specific form of Attention is called SelfAttention, since the input sequence attends to itself. It is a key component that allows the Encoder to encode the input sequence efficiently.

## Transformers for Q-learning Transformer-based Q-Networks

This paper proposes to use an altered version of the Transformer Encoder as a Q-network for a Q-learning agent. However, the original structure has to be altered slightly to be usable. To map to Q-values at the end of the model, the output of the Encoder has to be mapped to the Q-value dimension which is achieved by adding a fully connected layer after the last Encoder layer. Also, the embedding layer of the classic Transformer has to be replaced by a fully connected layer that maps from the state dimension to the model dimension. After these two steps, a Transformer-based Qnetwork (TBQN) that can map from states to Q-values is obtained. It can be examined in Figure 2.

The literature (Parisotto et al. 2019), (Upadhyay et al. 2019), (Mishra et al. 2017) suggests, that a Q-learning agent using the proposed TBQN would be very hard to optimize and most likely unstable. To preemptively counter this, a method variation search space was constructed which includes three categories. Firstly, changes to the model structure itself. Secondly, the application of additional methods for DQNs and Transformers. Both of these categories represent small model or method variations that are proposed in the literature and might be able to improve the performance of TBQNs. Thirdly, a selection of possible impactful Hyperparameters is included. This search space was then filtered to find a method variation that is easier to optimize and more stable than a base Q-learning agent featuring the base TBQN.

## Transformer layer variations

Since the original publication of the Transformer (Vaswani et al. 2017), many Transformer layer variations were introduced in the literature. These structural changes are exclusively made to make the Transformer more stable during training. From this literature, several Transformer layer variations were selected to be tested as the core layer for TBQNs.

Dropout free models (layer type 2) Since Transformers were initially developed for NLP, they feature the usage of Dropout layers. Typically implemented to counter overfitting, the usage of Dropout in RL is not popular. Due to this, a layer without Dropout was tested. Additionally, all layer variations are tested with and without Dropout after the final layer. This layer variation is displayed in Figure 3a.

Figure 2: The proposed Transformer-based Q-network

Identity Map Reordering (IMR) (layer type 3) A layer variation that was described in (Parisotto et al. 2019). It features the positional change of the normalization layer to the start of each sub-layer. Furthermore, an additional ReLU activation after every sub-layer was added to prevent two linear layers in a row. Its implementation can be observed in Figure 3b.

Pre layer Normalization (layer type 4) Very similar to IMR, this variation described in (Xiong et al. 2020) changes the position of the layer normalization to the beginning of each sub-layer. While this is identical to IMR, this variation does not feature an additional ReLU activation. Its implementation can be observed in Figure 3c.

Output gate connections (layer type 5) Also described in (Parisotto et al. 2019) this variation based on IMR additionally replaces the residual connection with a gated layer. While residual connections were initially implemented to improve the training of deep neural networks, they seem to make training Transformers more unstable. They are replaced with the following gate formulation, where W and b are trainable parameters:

<!-- formula-not-decoded -->

This variation was also already tested for Transformerbased methods in RL and it will be used as it was proposed in (Parisotto et al. 2019).

GRU gate connections (layer type 6) Finally, another variation will be tested which features the usage of a different gating mechanism based on a GRU unit. Again, this variation was introduced in (Parisotto et al. 2019) and is based on IMR. Noteworthy is that this model variation combined with Maximum a Posteriori Policy Optimization (Song et al.

Figure 3: The Transformer Encoder layer variations

2019) achieved SOTA results for DMLab-30. It remains to be seen if this is also the case for Q-learning. The mechanism is defined by Formula (6). W and U are trainable parameters.

<!-- formula-not-decoded -->

## Additional methods and Hyperparameters

Additionally to these layer variations, the following methods and Hyperparameters were included categorically in the search space to test their effect on the performance of a TBQN:

- Double Q-learning
- Target update period
- Target update ( τ ) (Lillicrap et al. 2015)
- Gradient Clipping
- Learning rate schedules
- Depth-Scaled Initialization (Zhang, Titov, and Sennrich 2019)
- Depth-Scaled Initialization of the last Layer (Zhang, Titov, and Sennrich 2019)
- Number of Attention Heads
- Initial collection steps
- Loss function
- Environment normalization
- Epsilon Greedy
- Replay Buffer size
- Future reward discount ( γ )
- Batch size
- Learning rate
- Encoder type (whether or not dropout is used outside of the Encoder layers)

## Experiments

## Baseline performance

To motivate the method variation search and to set a base performance of TBQNs, a Q-learning agent with the proposed base TBQN and with no special additions (except a Replay Buffer and a Target Network) was evaluated. The agent was trained on four environments (MountainCar-v0, Acrobot-v1, CartPole-v1, and LunarLander-v2) for 150k steps. All these environments are implemented by OpenAI GYM (Brockman et al. 2016). The average episode return over 10 episodes can be examined in Figure 4 . Two training runs per environment were executed (For Acrobot-v1, only one is displayed to guarantee the visibility of the results). The agent was not able to solve any environment sufficiently, had a high fluctuation, and even diverged on some occasions (denoted by a graph ending before 150k steps). For these experiments, the following Hyperparameters were used. Initial collect steps: 1000, mean squared loss, 4 Attention Heads, epsilon greedy: 0.1, Replay Buffer length: 100000, batch size: 32, learning rate: 1e-5. The rest of the parameters were not used. It shows quite clearly, that TBQNs need additional help to perform.

## Selecting the optimal method variation

While it would be ideal to test every possible method variation, this is unfeasible due to computational complexity. Due to that, a two-step method based on two distinct studies was constructed to find a well-performing method variation.

Study one - Parameter importance The first study focused on narrowing down the method search space significantly. This was achieved by estimating the Mean Decrease Impurity Importance Score for all parameters in the method search space. Based on these scores, parameters with low importance were excluded entirely. Furthermore, parameters with high importance were further evaluated to select the best performing values and exclude the rest from the search space. To estimate these scores, the method search space had to be sampled and evaluated. Since grid search was infeasible, a Tree-Structured Parzen Estimator, which was firstly described in (Bergstra et al. 2011), was used to sample from the search space. Every method search space sample was trained for 15k steps. As a final performance score, the average return of the last 10 episodes was used. To guarantee generality, the study was conducted independently in three different environments (CartPole-v1, Acrobot-v1, and LunarLander-v2) and the final importance score for every parameter was averaged between these environments. All studies were performed on a single GPU (Nvidia1080Ti). Further information can be found in Appendix B.

Figure 4: Average return during training of a base Q-learning agent with the proposed TBQN base in four different environments.

Study two - Final selection After having narrowed down the method search space significantly, the remaining search space samples were evaluated further to find the model variation with the best performance. Two methods were used to determine the effect of certain parameter values on the performance of TBQNs and to select a final method variation: On one hand, the mean reward of the last ten episodes between all samples where a certain parameter value was present was calculated. On the other hand, the search space samples with the highest rewards for every environment were extracted. This was done to determine whether combinations of specific parameter values performed especially well. Again, the study was conducted in three different environments (CartPole-v1, Acrobot-v1, and LunarLander-v2) to guarantee generality. All search space samples were trained for 75K steps in the environments CartPole-v1 and AcroBot-v1 and for 150k steps in the environment LunarLander-v2. All experiments were conducted on a single Nvidia GPU(1080ti). Further information can be found in Appendix B.

## Results

Based on the two studies, the method variation represented by Table 1 was selected as it performed well in all environments and proved to be stable during training. The parameters initial collect steps, Environment normalization, Replay Buffer size, τ , double Q-learning, and the Encoder type had low importance for control environments and are not specified. The following comments should be made to accompany this selection:

Table 1: Final method variation

| Parameter                                                                                                                                                                                                      | Value                                                                     | Category                                                                                                                                          |
|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------|
| Gradient Clipping Batch size Learning rate Layer type Custom lr schedule Depth-Scaled Initial- ization Target upate period Num Heads Epsilon Greedy Depth-Scaled Initial- ization (last layer) Loss function γ | True 32 1e-4 3 'No' 1 10+ 4/2 (0. - 1.) (T/F) (Huber, Squared) (.99, .95) | Fixed Fixed Fixed Fixed Fixed Fixed Semi-fixed Semi-fixed Environment dependent Environment dependent Environment dependent Environment dependent |

- The optimal values for several parameters are environment-dependent. This means the performance of a Q-learning agent using a TBQN relies strongly on the right value selection. The optimal values however change from environment to environment.
- Surprisingly, IMR layers (layer type 3) perform the best while GRU-gated layers (layer type 6) were excluded early due to frequent divergence.
- While being very important for NLP, learning rate schedules are not required for TBQNs. It is estimated that TBQNs with layer variations do not require learning rate schedules which makes them obsolete.
- Depth-Scaled Initialization (Zhang, Titov, and Sennrich 2019) is beneficial. Models that were initialized with it tended to diverge less and achieved higher average rewards at the end of training.
- Gradient Clipping is very important for TBQNs. Since the Transformer has problems with divergence in the RL setting, Gradient Clipping helps to mitigate destructive updates.
- Several parameters are not important for model performance (Assuming no abstruse values). During the parameter search, they showed no significant impact on the performance of TBQNs.

## Performance in control environments

To evaluate the performance of the method variation specified in table 1, its average return during training was compared to the average return during training of an optimized classic Q-learning agent in four Environments (CartPole-v1, Acrobot-v1, MountainCar-v0, and LunarLander-v2). The method was extracted from Rl-zoo baselines (Raffin 2018), a collection of Hyperparameter optimized methods. Additionally, the final method variation was tested with different values for history length, model dimensions, and the number of layers (Appendix C) to secure that it is stable and performs consistently when scaled up or down. By examining Figure 5, it is visible that the performance of the proposed model is consistent over different model sizes. While the method variation is consistent for CartPole-v1, only one variation is displayed to keep the visibility of the results. Furthermore, when comparing Figure 5 with Figure 6, it is visible that the average return during training of these different approaches is largely comparable. Noteworthy, the proposed method variation seems to have problems to converge for CartPole-v1. The frequency in which the maximum reward is achieved seems to however increase over time.

Figure 5: Average return during training of the final model variation with different model dimensions on control environments

## Performance in ATARI environments

Additionally to the control environments, the proposed method variation was trained in two environments ('MsPacman' and 'Asteroids') of the popular ATARI benchmark. Two parameters that are not included in 1 were scaled up from their initial values to match the complexity of the new environment and to keep them in reasonable ranges. The initial collect steps were increased from 1000 to 5000. Additionally, the Replay Buffer size was increased from 100k to 200k. For both environments, the RAM state which consists out of 128 pixels was used as the state vector for training. The proposed method was trained on MsPacman for 4 and 5 million timesteps and on 'Asteroids' on 5 million timesteps. The best average return during training was compared to the reported results from (Mnih et al. 2015) (classic DQN performance), and (Hausknecht and Stone 2015) (RQN performance). Both studies trained their methods for 10 million timesteps before reporting their final average returns.

When tracking the method state with the best average return during training for the 'Asteroids' environment, the proposed method variation performs quite well. After only 5 million timesteps, the Q-learning agent achieved a higher average return than the reported RQN-based and DQN-based methods. However, during training, the model performance fluctuates strongly which makes the final Q-learning agent perform quite bad. For the 'MsPacman' environment, no superior performance can be reported. Additionally, the training behavior does vary significantly. The same TBQN-based Q-learning agent was trained twice. The first try (7b) shows consistent learning over the whole training period. The second one (7c) shows no increase in performance over the whole training. While TBQN based methods can perform in ATARI environments, it is still a challenging task. More experiments must be conducted to form a final conclusion. It is also suspected that conducting the parameter search in control environments, might have had a negative effect on the performance of TBQNs in ATARI environments. We are positive, that this challenge can however be overcome by committing more computational resources in the future.

Figure 6: Average return during training of a HP optimized classic Q-learning agent

Figure 7: Average return during training on Atari

Table 2: Reported average returns of different methods on Atari. 1 = (Hausknecht and Stone 2015), 2 = (Mnih et al. 2015)

| Methods                | Asteroids                                       | MsPacman                                       |
|------------------------|-------------------------------------------------|------------------------------------------------|
| DQN 2 DQN 1 RQN 1 TBQN | 1629 +/- 542 1070+/-345 1020 +/-312 1813+/- 396 | 2311 +/- 525 2363 +/-735 2048+/-653 1555+/-696 |

## Conclusion

During this work, the interaction of Transformer architectures and Deep Q-learning was evaluated. The goal of this work was to craft a new RL method based on the combination of Deep Q-learning and Transformer-based models which was successful. Through an extensive method variation search, a Transformer-based Deep Q-Learning method was constructed which leverages developments around Transformers as well as Q-learning. The proposed model can match the performance of an optimized classic Q-learning agent on control environments while showing potential on selected Atari environments. Despite these successes, the testing of the proposed final method variation on more environments and especially environments that require a deep understanding of past states is still essential to form a final conclusion. The results of this work are complementary to (Parisotto et al. 2019) and another step to a better understanding of Transformer architectures in RL. This work defies past results that neglect Transformer architectures in RL and shows that they can perform when handled carefully. While the proposed method is connected to the one that was used in (Parisotto et al. 2019), it represents a different version of a Transformer-based RL method that can be deployed, tuned, and tested more easily. To further encourage this, the code base of this research can be accessed under (Stein 2020). It is hoped that this work can help to support new studies on the topic of Transformers in RL and leverage them to RL mainstream.

## Appendix

## A. Model specifications

Throughout this work, the TBQN dimensions specified in Table 3 were used.

Table 3: TBQN dimensions throughout this work

| Specification                                                     | Control          | Atari            |
|-------------------------------------------------------------------|------------------|------------------|
| History horizon Encoding Dimension Number of Layers Dff Dimension | 5 steps 64 3 256 | 4 steps 64 2 256 |

## B. Study specifications

Table 4 and Table 5 hold additional information concerning the studies that were conducted to arrive at a final method variation.

Table 4: Additional information for study 1

| Specification                                                                                  | Value      |
|------------------------------------------------------------------------------------------------|------------|
| Number of evaluated search space samples Number of environments Runs per sample Training steps | 30 3 2 15k |

Table 5: Additional information for study 2

| Specification                                                                        | Value         |
|--------------------------------------------------------------------------------------|---------------|
| Remaining search space samples Number of environments Runs per sample Training steps | 24 3 2 150k / |

## C. Model dimension variants

During the evaluation of the final TBQN variation, the model dimensions were altered to test for stability when scaling TBQNs up or down. The following variations were tested:

- History horizon: 5, Dimensions: 64/256, Layers: 3

- History horizon: 5, Dimensions: 64/256, Layers: 6

- History horizon: 3, Dimensions: 64/256, Layers: 3

- History horizon: 7, Dimensions: 64/256, Layers: 3

- History horizon: 5, Dimensions: 128/512, Layers: 3

## D. Additional comments

All experiments and studies were conducted on a single GPU (Nvidia1080Ti). Specific parameters that are not explicitly defined are set to the default values of TensorFlow (Abadi et al. 2015) or are defined in the experiment scripts available at (Stein 2020)

## References

Abadi, M.; Agarwal, A.; Barham, P.; Brevdo, E.; Chen, Z.; Citro, C.; Corrado, G. S.; Davis, A.; Dean, J.; Devin, M.; Ghemawat, S.; Goodfellow, I.; Harp, A.; Irving, G.; Isard, M.; Jia, Y.; Jozefowicz, R.; Kaiser, L.; Kudlur, M.; Levenberg, J.; Man´ e, D.; Monga, R.; Moore, S.; Murray, D.; Olah, C.; Schuster, M.; Shlens, J.; Steiner, B.; Sutskever, I.; Talwar, K.; Tucker, P.; Vanhoucke, V.; Vasudevan, V.; Vi´ egas, F.; Vinyals, O.; Warden, P.; Wattenberg, M.; Wicke, M.; Yu, Y.; and Zheng, X. 2015. TensorFlow: Large-Scale Machine Learning on Heterogeneous Systems. URL http: //tensorflow.org/. Software available from tensorflow.org.

Bergstra, J. S.; Bardenet, R.; Bengio, Y.; and K´ egl, B. 2011. Algorithms for hyper-parameter optimization. In Advances in neural information processing systems , 2546-2554.

Brockman, G.; Cheung, V.; Pettersson, L.; Schneider, J.; Schulman, J.; Tang, J.; and Zaremba, W. 2016. OpenAI Gym.

Dai, Z.; Yang, Z.; Yang, Y.; Carbonell, J.; Le, Q. V.; and Salakhutdinov, R. 2019. Transformer-xl: Attentive language models beyond a fixed-length context. arXiv preprint arXiv:1901.02860 .

Devlin, J.; Chang, M.-W.; Lee, K.; and Toutanova, K. 2018. Bert: Pre-training of deep bidirectional transformers for language understanding. arXiv preprint arXiv:1810.04805 .

Hausknecht, M.; and Stone, P. 2015. Deep recurrent qlearning for partially observable mdps. In 2015 AAAI Fall Symposium Series .

Iqbal, S.; and Sha, F. 2019. Actor-attention-critic for multiagent reinforcement learning. In International Conference on Machine Learning , 2961-2970. PMLR.

Lillicrap, T. P.; Hunt, J. J.; Pritzel, A.; Heess, N.; Erez, T.; Tassa, Y.; Silver, D.; and Wierstra, D. 2015. Continuous control with deep reinforcement learning. arXiv preprint arXiv:1509.02971 .

Manchin, A.; Abbasnejad, E.; and van den Hengel, A. 2019. Reinforcement learning with attention that works: A selfsupervised approach. In International Conference on Neural Information Processing , 223-230. Springer.

Mishra, N.; Rohaninejad, M.; Chen, X.; and Abbeel, P. 2017. A simple neural attentive meta-learner. arXiv preprint arXiv:1707.03141 .

Mnih, V.; Kavukcuoglu, K.; Silver, D.; Graves, A.; Antonoglou, I.; Wierstra, D.; and Riedmiller, M. 2013. Playing atari with deep reinforcement learning. arXiv preprint arXiv:1312.5602 .

Mnih, V.; Kavukcuoglu, K.; Silver, D.; Rusu, A. A.; Veness, J.; Bellemare, M. G.; Graves, A.; Riedmiller, M.; Fidjeland, A. K.; Ostrovski, G.; et al. 2015. Human-level control through deep reinforcement learning. Nature 518(7540): 529-533.

Oh, J.; Chockalingam, V.; Singh, S.; and Lee, H. 2016. Control of memory, active perception, and action in minecraft. arXiv preprint arXiv:1605.09128 .

Parisotto, E.; Song, H. F.; Rae, J. W.; Pascanu, R.; Gulcehre, C.; Jayakumar, S. M.; Jaderberg, M.; Kaufman, R. L.; Clark, A.; Noury, S.; et al. 2019. Stabilizing Transformers for Reinforcement Learning. arXiv preprint arXiv:1910.06764 .

Radford, A.; Wu, J.; Child, R.; Luan, D.; Amodei, D.; and Sutskever, I. 2019. Language models are unsupervised multitask learners. OpenAI Blog 1(8): 9.

Raffin, A. 2018. RL Baselines Zoo. URL https://github. com/araffin/rl-baselines-zoo.

Song, H. F.; Abdolmaleki, A.; Springenberg, J. T.; Clark, A.; Soyer, H.; Rae, J. W.; Noury, S.; Ahuja, A.; Liu, S.; Tirumala, D.; et al. 2019. V-MPO: On-Policy Maximum a Posteriori Policy Optimization for Discrete and Continuous Control. arXiv preprint arXiv:1909.12238 .

Stein, G. 2020. Gideon-Stein/TBQN. URL https://github. com/Gideon-Stein/TBQN.

Upadhyay, U.; Shah, N.; Ravikanti, S.; and Medhe, M. 2019. Transformer Based Reinforcement Learning For Games. arXiv preprint arXiv:1912.03918 .

Vaswani, A.; Shazeer, N.; Parmar, N.; Uszkoreit, J.; Jones, L.; Gomez, A. N.; Kaiser, Ł.; and Polosukhin, I. 2017. Attention is all you need. In Advances in neural information processing systems , 5998-6008.

Xiong, R.; Yang, Y.; He, D.; Zheng, K.; Zheng, S.; Xing, C.; Zhang, H.; Lan, Y.; Wang, L.; and Liu, T.-Y. 2020. On layer normalization in the transformer architecture. arXiv preprint arXiv:2002.04745 .

Zhang, B.; Titov, I.; and Sennrich, R. 2019. Improving deep transformer with depth-scaled initialization and merged attention. arXiv preprint arXiv:1908.11365 .

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [18]:
text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [19]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

splitter = CharacterTextSplitter(
    chunk_size= 60,
    chunk_overlap= 10,
    # chunk_overlap은 separator가 빈 문자열일때 적용된다.
    separator="" # 기본: `\n\n` -> "" 변경. "": chunk_size에 맞추겠다.
)

result = splitter.split_text(text) # 입력이 str일때 이걸사용
print(type(result))
print(type(result[0]))
print(len(result))

Created a chunk of size 69, which is longer than the specified 60


<class 'list'>
<class 'str'>
4


In [20]:
for chunk in result:
    print(len(chunk), chunk)
    print("----------------------------------------------")

69 123456789012345678901234567890123456789012345678901234567890123456789
----------------------------------------------
52 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
----------------------------------------------
26 가나다라마바사아자차카타파하

아야어여오요우유으이
----------------------------------------------
52 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
----------------------------------------------


In [21]:
doc = Document(page_content= text)
# 나눈 문서가 Document 객체일때
result_docs = splitter.split_documents([doc])

Created a chunk of size 69, which is longer than the specified 60


In [22]:
print(len(result_docs))
print(type(result_docs[0]))

4
<class 'langchain_core.documents.base.Document'>


In [23]:
for doc in result_docs:
    print(doc)
    print("----------------------------------")

page_content='123456789012345678901234567890123456789012345678901234567890123456789'
----------------------------------
page_content='abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
----------------------------------
page_content='가나다라마바사아자차카타파하

아야어여오요우유으이'
----------------------------------
page_content='abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
----------------------------------


In [ ]:
text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST.UVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''], # 구분자 리스트를 직접 지정
    is_separator_regex=True # 구분자에 정규표현식 사용가능 여부.
)

result2 = splitter.split_text(text2)

In [41]:
for txt in result2:
    print(len(txt),txt)
    print("--------------------------------------------------------------")

40 1234567890123456789012345678901234567890
--------------------------------------------------------------
29 12345678901234567890123456789
--------------------------------------------------------------
49 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
--------------------------------------------------------------
13 NOPQRSTUVWXYZ
--------------------------------------------------------------
26 가나다라마바사아자차카타파하

아야어여오요우유으이
--------------------------------------------------------------
43 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ
--------------------------------------------------------------
9 RSTUVWXYZ
--------------------------------------------------------------
49 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
--------------------------------------------------------------
13 NOPQRSTUVWXYZ
--------------------------------------------------------------


In [27]:
print(splitter._separators)

['\n\n', '\n', ' ', '']


In [46]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

path = "data/olympic.txt"

# load
loader = TextLoader(path, encoding="UTF-8")
docs = loader.load()
print("Load한 문서 개수", len(docs))

# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap= 50,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''], # 구분자 리스트를 직접 지정
    is_separator_regex= True
)
split_docs = splitter.split_documents(docs)
print("Split후 문서 개수", len(split_docs))

Load한 문서 개수 1
Split후 문서 개수 61


In [47]:
split_docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [48]:
# 문서 Load와 Split을 한번에 처리
docs = loader.load_and_split(splitter)
print(len(docs))

61


In [50]:
docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `o200k_base`: GPT-4 이후 모델들이 사용한 방식
    - `cl100k_base`: 초기 GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [56]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

loader = TextLoader(path, encoding="utf-8")
# from_tiktoken_encoder() -> OpenAI GPT모델을 사용할 경우. 이걸 사용하는 것이 좋다.
#                            GPT모델명 또는 Encoder(토크나이저)의 이름
# splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     # model_name="gpt-5",
#     encoding_name="o200k_base",
#     chunk_size= 500, # 500토큰 기준
#     chunk_overlap= 50, # 50토큰
# )
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # model_name="gpt-5",
    encoding_name="o200k_base",
    chunk_size= 500, # 500토큰 기준
    chunk_overlap= 50, # 50토큰
)

docs = loader.load_and_split(splitter)
print(len(docs))

Created a chunk of size 1027, which is longer than the specified 500
Created a chunk of size 981, which is longer than the specified 500
Created a chunk of size 881, which is longer than the specified 500
Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 703, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 925, which is longer than the specified 500
Created a chunk of size 661, which is longer than the specified 500
Created a chunk of size 1305, which is longer than the specified 500
Created a chunk of size 1052, which is longer than the specified 500


18


In [57]:
docs[0].page_content

'올림픽\n올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.\n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제

In [58]:
print(len(docs[0].page_content))

1726


In [62]:
from transformers import AutoTokenizer

model_id="google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id) # model_id는 사용할 LLM 모델의 이름

# huggingface tokenizer 이용
splitter_hf = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer= tokenizer, # transformers.Tokenizer객체를 넣어줘야함.
    chunk_size= 500,
    chunk_overlap= 50,
)

c:\SKN31_\SKN31\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--google--gemma-3-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [64]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2
## 중주제2
- 기차
- 배
"""

In [71]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 나눌 때 기준이 되는 Header를 설정.
# dict: key-Header기호, value: 이름
header_to_split = [
    ("#", "header1"),
    ("##", "header2"),
    ("###", "header3"),
    ("####", "header4"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on= header_to_split,
    strip_headers= False, # default: True - 구분자 Header(제목)을 내용에 표시할 지 여부. True: 안넣는다.
)

docs = splitter.split_text(text)
len(docs)

4

In [72]:
docs

[Document(metadata={'header1': '대주제1'}, page_content='# 대주제1\n- 동물'),
 Document(metadata={'header1': '대주제1', 'header2': '중주제1'}, page_content='## 중주제1\n- 포유류  \n- 조류'),
 Document(metadata={'header1': '대주제1', 'header2': '중주제1', 'header3': '소주제1'}, page_content='### 소주제1\n- 개\n- 고양이\n- 까치\n- 독수리'),
 Document(metadata={'header1': '대주제2', 'header2': '중주제2'}, page_content='# 대주제2  \n## 중주제2\n- 기차\n- 배')]

In [74]:
# olympic.md

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/olympic_wiki.md"
loader = TextLoader(path, encoding="utf-8")
header_to_split = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = header_to_split
)

In [75]:
docs = loader.load()

# list[Document] -> Document에서 page_content(읽은 text)를 조회해서 하나의 str로 반환
doc_txt = "\n".join(doc.page_content for doc in docs)

# MarkdownHeaderSplitter는 split_document가 없고 split_text(str)만 제공
split_docs = splitter.split_text(doc_txt)
len(split_docs)

25

In [77]:
print(split_docs[0].page_content)

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  
또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제, 정치,